In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import Counter
import random
import numpy as np
from pathlib import Path

In [21]:
# -----------------------------
# 1. Sample corpus
# -----------------------------

#corpus = [
#    "A mother loves her child",
#    "A father loves his child",
#    "Parents care for their children",
#    "A mother and father are parents",
#    "Children love their mother",
#    "Children love their father",
#]

#prepare the data
!python prep_fairytales.py --indir ./raw_fairytales --outdir corpus --lower --dedup

Loaded raw_fairytales\7439-h.htm
{
  "stories": 1,
  "tokens": 61026,
  "vocab": 5658,
  "removed_non_english": 0,
  "generated_at": "2026-02-18T14:07:37.972454Z"
}
✔ Wrote 1 stories to corpus\clean


In [22]:
# -----------------------------
# 2. Preprocessing
# -----------------------------
#def tokenize_corpus(corpus): old corpus tokenizer
#    tokens = [sentence.lower().split() for sentence in corpus]
#    return tokens

def iter_tokens(tokens_path: Path):
    with tokens_path.open("r", encoding="utf-8") as f:
        for line in f:
            yield line.strip().split()

In [23]:
tokens_path = Path('.\\corpus\\clean\\corpus.txt')
tokenized_corpus = list(iter_tokens(tokens_path))

#tokenized_corpus = tokenize_corpus(corpus) ## old corpus tokenizer
vocab = Counter([word for sentence in tokenized_corpus for word in sentence])
word2idx = {word: idx for idx, (word, _) in enumerate(vocab.items())}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)

In [5]:
word2idx

{'text': 0,
 'file': 1,
 'produced': 2,
 'by': 3,
 'charles': 4,
 'franks,': 5,
 'delphine': 6,
 'lettau': 7,
 'and': 8,
 'the': 9,
 'people': 10,
 'at': 11,
 'dp': 12,
 'html': 13,
 'david': 14,
 'widger': 15,
 'english': 16,
 'fairy': 17,
 'tales': 18,
 'anonymous': 19,
 'collected': 20,
 'joseph': 21,
 'jacobs': 22,
 'how': 23,
 'to': 24,
 'get': 25,
 'into': 26,
 'this': 27,
 'book.': 28,
 'knock': 29,
 'knocker': 30,
 'on': 31,
 'door,': 32,
 'pull': 33,
 'bell': 34,
 'side,': 35,
 'then,': 36,
 'if': 37,
 'you': 38,
 'are': 39,
 'very': 40,
 'quiet,': 41,
 'will': 42,
 'hear': 43,
 'a': 44,
 'teeny': 45,
 'tiny': 46,
 'voice': 47,
 'say': 48,
 'through': 49,
 'grating': 50,
 '"take': 51,
 'down': 52,
 'key."': 53,
 'find': 54,
 'back:': 55,
 'cannot': 56,
 'mistake': 57,
 'it,': 58,
 'for': 59,
 'it': 60,
 'has': 61,
 'j.': 62,
 'in': 63,
 'wards.': 64,
 'put': 65,
 'key': 66,
 'keyhole,': 67,
 'which': 68,
 'fits': 69,
 'exactly,': 70,
 'unlock': 71,
 'door': 72,
 'walk': 73,
 '

In [24]:
vocab_size

9147

In [25]:
vocab

Counter({'the': 4056,
         'and': 3134,
         'to': 1611,
         'a': 1306,
         'of': 1230,
         'he': 1068,
         'in': 815,
         'was': 706,
         'she': 680,
         'his': 646,
         'that': 533,
         'it': 503,
         'her': 461,
         'for': 445,
         'you': 434,
         'with': 430,
         'so': 410,
         'as': 398,
         'they': 391,
         'i': 379,
         'on': 375,
         'but': 375,
         'at': 325,
         'had': 311,
         'said': 309,
         'is': 282,
         'all': 276,
         'him': 273,
         'went': 270,
         'be': 264,
         'my': 245,
         'when': 243,
         'by': 240,
         'have': 233,
         'little': 227,
         'up': 217,
         'then': 212,
         'out': 209,
         'not': 209,
         'came': 205,
         'this': 194,
         'down': 183,
         'one': 181,
         'there': 181,
         ',': 177,
         'jack': 164,
         'into': 158,
         

In [26]:
# -----------------------------
# 3. Generate training data (Skip-gram)
# -----------------------------
def generate_skipgram_data(tokenized_corpus, window_size=4):
    pairs = []
    for sentence in tokenized_corpus:
        for center_idx, center_word in enumerate(sentence):
            context_range = range(max(0, center_idx - window_size),
                                  min(len(sentence), center_idx + window_size + 1))
            for context_idx in context_range:
                if context_idx != center_idx:
                    pairs.append((center_word, sentence[context_idx]))
    return pairs

pairs = generate_skipgram_data(tokenized_corpus)

# Convert to indices
training_data = [(word2idx[w1], word2idx[w2]) for w1, w2 in pairs]


In [27]:
# -----------------------------
# 4. Negative Sampling
# -----------------------------
def get_negative_samples(pos_word_idx, num_negatives=5):
    neg_samples = []
    #random.seed(0)
    while len(neg_samples) < num_negatives:
        neg = random.randint(0, vocab_size - 1)
        if neg != pos_word_idx:
            neg_samples.append(neg)
    return neg_samples


In [28]:
# -----------------------------
# 5. Word2Vec Model
# -----------------------------
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(Word2Vec, self).__init__()
        #torch.manual_seed(0)
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)

    def forward(self, context_words, pos_context, neg_context):
        center_embeds = self.in_embed(context_words)
        pos_embeds = self.out_embed(pos_context)
        neg_embeds = self.out_embed(neg_context)

        # Positive score
        pos_score = F.cosine_similarity(pos_embeds, center_embeds,dim=1)
        pos_loss = torch.log(torch.sigmoid(pos_score) + 1e-10)

        # Negative score
        neg_score =  F.cosine_similarity(neg_embeds, center_embeds.unsqueeze(1), dim=2)
        neg_loss = torch.log(1-torch.sigmoid(neg_score) + 1e-10)

        return -(pos_loss.mean()+neg_loss.mean())/2
            
    def get_in_embeddings(self):
        return self.in_embed.weight.data.cpu()
    def get_out_embeddings(self):
        return self.out_embed.weight.data.cpu()



In [11]:
### Example just two batches for demonstration in excel

embedding_dim = 4
model = Word2Vec(vocab_size, embedding_dim)
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 1
batch_size = 3
num_negatives = 2


total_loss = 0

for i in range(0,2):
    batch = training_data[i:i+batch_size]
    print("batch: ",str(i))
    print(batch)
    center_batch = torch.LongTensor([c for c, _ in batch])
    pos_batch = torch.LongTensor([p for _, p in batch])
    neg_batch = torch.LongTensor([get_negative_samples(p, num_negatives) for _, p in batch])

    embedings_in = model.get_in_embeddings()
    embedings_out = model.get_out_embeddings()

    optimizer.zero_grad()

    center_embeds = embedings_in[center_batch]  # (batch, embed_dim)
    pos_embeds = embedings_out[pos_batch]    # (batch, embed_dim)
    neg_embeds = embedings_out[neg_batch] 
    
    print("center_batch:",center_batch)
    print(center_embeds)
    print("pos_batch:",pos_batch)
    print(pos_embeds)
    print("neg_batch:",neg_batch)
    print(neg_embeds)
    
     # Positive score
    pos_score = F.cosine_similarity(pos_embeds, center_embeds,dim=1)
    pos_loss = torch.log(torch.sigmoid(pos_score) + 1e-10)
    
    print("pos_score:",pos_score)
    print("sigmoid:",torch.sigmoid(pos_score))
    print("pos_loss:",pos_loss)
    # Negative score
    neg_score = F.cosine_similarity(neg_embeds, center_embeds.unsqueeze(1), dim=2)
    neg_loss = torch.log(1-torch.sigmoid(neg_score) + 1e-10)

    print("neg_score:",neg_score)
    print("sigmoid:",torch.sigmoid(neg_score))
    print("neg_loss:",neg_loss)

    loss = model(center_batch, pos_batch, neg_batch)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

    print("center_batch:",center_batch)
    print(embedings_in[center_batch])
    print("pos_batch:",pos_batch)
    print(embedings_out[pos_batch])
    print("neg_batch:",neg_batch)
    print(embedings_out[neg_batch])

batch:  0
[(0, 1), (0, 2), (0, 3)]
center_batch: tensor([0, 0, 0])
tensor([[ 0.3433, -0.6791, -1.7486, -0.0395],
        [ 0.3433, -0.6791, -1.7486, -0.0395],
        [ 0.3433, -0.6791, -1.7486, -0.0395]])
pos_batch: tensor([1, 2, 3])
tensor([[ 0.0219, -0.6866, -0.7662, -0.8344],
        [ 0.7766, -1.5121,  1.6752, -2.2265],
        [ 1.8788,  0.4637, -0.3432,  0.4902]])
neg_batch: tensor([[ 655, 4677],
        [2807,  906],
        [2790, 5529]])
tensor([[[ 1.2102, -0.9275, -0.5694, -1.4435],
         [-0.7706, -0.9451, -0.6896, -0.2577]],

        [[ 2.8023,  1.8299, -0.4839,  1.6852],
         [-1.5911,  0.3237,  1.2516, -0.5820]],

        [[ 0.3720, -0.9690,  0.1758,  0.0684],
         [-0.4193, -0.1842,  0.5037,  0.7293]]])
pos_score: tensor([ 0.7307, -0.2486,  0.2357])
sigmoid: tensor([0.6750, 0.4382, 0.5587])
pos_loss: tensor([-0.3931, -0.8252, -0.5822])
neg_score: tensor([[ 0.5056,  0.5864],
        [ 0.0692, -0.7212],
        [ 0.2364, -0.4879]])
sigmoid: tensor([[0.6238, 0.6

In [12]:
# -----------------------------
# 6. Training
# -----------------------------
embedding_dim = 200
model = Word2Vec(vocab_size, embedding_dim)
optimizer = optim.Adam(model.parameters(), lr=0.01)

epochs = 50
batch_size = 50
num_negatives = 2

epoch_loss = []

for epoch in range(epochs):
    print("Epoch:",epoch+1)
    total_loss = 0
    #random.shuffle(training_data)
    for i in range(0, len(training_data), batch_size):
        batch = training_data[i:i+batch_size]
        center_batch = torch.LongTensor([c for c, _ in batch])
        pos_batch = torch.LongTensor([p for _, p in batch])
        neg_batch = torch.LongTensor([get_negative_samples(p, num_negatives) for _, p in batch])

        optimizer.zero_grad()
        loss = model(center_batch, pos_batch, neg_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    epoch_loss.append(total_loss)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")
torch.save(model, "word2vect_tania_old.pt")

Epoch: 1
Epoch 1/50, Loss: 5491.6804
Epoch: 2
Epoch 2/50, Loss: 4935.7951
Epoch: 3
Epoch 3/50, Loss: 4842.1192
Epoch: 4
Epoch 4/50, Loss: 4798.6589
Epoch: 5
Epoch 5/50, Loss: 4777.5317
Epoch: 6
Epoch 6/50, Loss: 4765.0335
Epoch: 7
Epoch 7/50, Loss: 4756.3735
Epoch: 8
Epoch 8/50, Loss: 4750.4229
Epoch: 9
Epoch 9/50, Loss: 4750.3381
Epoch: 10
Epoch 10/50, Loss: 4746.5229
Epoch: 11
Epoch 11/50, Loss: 4743.8618
Epoch: 12
Epoch 12/50, Loss: 4743.0857
Epoch: 13
Epoch 13/50, Loss: 4744.8845
Epoch: 14
Epoch 14/50, Loss: 4742.1444
Epoch: 15
Epoch 15/50, Loss: 4742.9713
Epoch: 16
Epoch 16/50, Loss: 4740.0615
Epoch: 17
Epoch 17/50, Loss: 4739.2783
Epoch: 18
Epoch 18/50, Loss: 4739.6897
Epoch: 19
Epoch 19/50, Loss: 4739.8527
Epoch: 20
Epoch 20/50, Loss: 4738.8367
Epoch: 21
Epoch 21/50, Loss: 4737.7391
Epoch: 22
Epoch 22/50, Loss: 4738.1509
Epoch: 23
Epoch 23/50, Loss: 4737.9772
Epoch: 24
Epoch 24/50, Loss: 4734.8424
Epoch: 25
Epoch 25/50, Loss: 4736.7546
Epoch: 26
Epoch 26/50, Loss: 4736.4969
Epoc

In [13]:
#plotting the loss curve
import matplotlib.pyplot as plt
plt.plot(epoch_loss)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')

NameError: name 'epoch_loss' is not defined

In [ ]:
# -----------------------------
# 7. View learned embeddings for some words
# -----------------------------

#embeddings = model.get_embeddings()
#for word, idx in word2idx.items():
#    print(f"{word}: {embeddings[idx]}")

embeddings = model.get_embeddings()
# Example: Get embedding for a specific word
words = ["father","mother"]
for word in words:
    if word in word2idx:
        word_idx = word2idx[word]
        embedding_vector = embeddings[word_idx]
        print(f"Embedding for '{word}': {embedding_vector}")


In [29]:
model = torch.load("word2vect_tania_old.pt",map_location="cpu",weights_only=False)

In [30]:
embeddings = model.get_in_embeddings()

def find_similar_embedding(positive, negative, top_n=5):
    positive_negative = positive + negative
    cos = nn.CosineSimilarity(dim=0, eps=1e-6)

    for item in positive_negative:
        if item not in word2idx:
            print(f"{item} not in vocabulary")

    query_vec = embeddings[word2idx[positive[0]]] - embeddings[word2idx[negative[0]]] + embeddings[word2idx[positive[1]]]
    
    # Compute cosine similarity with all other words
    similarities = {}
    for word, idx in word2idx.items():
        if word not in positive_negative:
            
            sim = cos(query_vec,embeddings[idx])
            similarities[word] = sim
    
    # Sort by similarity
    similar_words = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    print(f"\nTop {top_n} words similar to embedding:")
    for word, score in similar_words:
        print(f"{word}: {score:.4f}")

In [32]:
find_similar_embedding(positive = ['king','woman'], negative = ['man'], top_n=5)


Top 5 words similar to embedding:
hill: 0.9943
green: 0.9932
one: 0.9931
hills: 0.9926
old,: 0.9926


In [38]:
from numpy.linalg import norm
cos = nn.CosineSimilarity(dim=0, eps=1e-6)

def find_similar_words(query_word, top_n=5):
    if query_word not in word2idx:
        print(f"'{query_word}' not in vocabulary.")
        return
    
    # Get the embedding for the query word
    query_idx = word2idx[query_word]
    query_vec = embeddings[query_idx]
    
    # Compute cosine similarity with all other words
    similarities = {}
    for word, idx in word2idx.items():
        if word != query_word:
            sim = cos(query_vec,embeddings[idx])
            similarities[word] = sim
    
    # Sort by similarity
    similar_words = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:top_n]
    
    print(f"\nTop {top_n} words similar to '{query_word}':")
    for word, score in similar_words:
        print(f"{word}: {score:.4f}")


In [41]:
# Example usage:
find_similar_words("king", top_n=5)


Top 5 words similar to 'king':
who: 0.9993
there: 0.9986
was: 0.9981
little: 0.9978
only: 0.9975


In [42]:
# Example usage:
find_similar_words("mother", top_n=5)


Top 5 words similar to 'mother':
last: 0.9991
soon: 0.9990
got: 0.9990
could: 0.9990
morning: 0.9990
